In [1]:
# =============================================================================
#  SHROOM-visions 2026 — text-only span tagger
#  XLM-R token classification -> per-character hallucination probability
#
#  Kaggle: T4, ~20 min end to end. Expects the data already extracted at
#  /kaggle/working/distrib/ (as in the earlier notebook).
#
#  Produces: dev scores + predictions_{lang}.jsonl.
# =============================================================================

In [2]:
ls /kaggle/working/distrib

ls: cannot access '/kaggle/working/distrib': No such file or directory


In [3]:
# ── Cell 0: fetch data if not already present ────────────────────────────────
import os
if not os.path.exists("/kaggle/working/distrib"):
    !wget -q https://a3s.fi/mickusti-2007780-pub/shroom-visions-data.zip -O /tmp/data.zip
    !unzip -oq /tmp/data.zip -d /kaggle/working/
print(sorted(os.listdir("/kaggle/working/distrib")))

['shroom-vision.test.en.unlabeled.jsonl', 'shroom-vision.test.fr.unlabeled.jsonl', 'shroom-vision.test.it.unlabeled.jsonl', 'shroom-vision.test.zh.unlabeled.jsonl', 'shroom-vision.train.en.labeled.jsonl', 'shroom-vision.train.fr.labeled.jsonl', 'shroom-vision.train.it.labeled.jsonl', 'shroom-vision.train.zh.labeled.jsonl']


In [4]:
# ── Cell 1: setup ────────────────────────────────────────────────────────────
import json, os, random, pathlib
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from scipy.stats import spearmanr

DISTRIB    = "/kaggle/working/distrib"
OUT_DIR    = "/kaggle/working"
MODEL_ID   = "xlm-roberta-large"
LANGS      = ["en", "fr", "it", "zh"]
CATEGORIES = ["invention", "mischaracterization", "OCR", "miscounting", "other"]
MAX_LEN    = 256
BATCH      = 8
EPOCHS     = 5
LR         = 1e-5
SEED       = 13

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

device: cuda


In [5]:
# ── Cell 2: official scoring functions ───────────────────────────────────────
# Copied verbatim from the organizers' scorer.py so dev numbers match the
# leaderboard exactly. Do not edit.

def score_cor(ref_dict, pred_dict, label_filtered_=None):
    assert ref_dict['id'] == pred_dict['id']
    ref_vec = [0.] * ref_dict['text_len']
    pred_vec = [0.] * ref_dict['text_len']
    ref_labels = (ref_dict['labels'] if label_filtered_ is None
                  else [s for s in ref_dict['labels'] if s['label'] == label_filtered_])
    pred_labels = (pred_dict['labels'] if label_filtered_ is None
                   else [s for s in pred_dict['labels'] if s['label'] == label_filtered_])
    for span in ref_labels:
        for idx in range(span['start'], span['end']):
            ref_vec[idx] += span['prob']
    for span in pred_labels:
        for idx in range(span['start'], span['end']):
            pred_vec[idx] = span['prob']
    ref_cmps = {round(f, 8) for f in ref_vec}
    pred_cmps = {round(f, 8) for f in pred_vec}
    if len(pred_cmps) == 1 or len(ref_cmps) == 1:
        if len(pred_cmps) != len(ref_cmps):
            return 0.0
        if ref_cmps == {0.0}:
            return float(pred_cmps == {0.0})
        return float(pred_cmps != {0.0})
    return spearmanr(ref_vec, pred_vec).correlation


def score_cor_lbl(ref_dict, pred_dict):
    all_labels = {s['label'] for d in [ref_dict, pred_dict] for s in d['labels']}
    if all_labels:
        return sum(score_cor(ref_dict, pred_dict, label_filtered_=l)
                   for l in all_labels) / len(all_labels)
    return 1.0


def score_iou(ref_dict, pred_dict):
    assert ref_dict['id'] == pred_dict['id']
    ref_i = {i for s in ref_dict['labels'] for i in range(s['start'], s['end'])}
    pred_i = {i for s in pred_dict['labels'] for i in range(s['start'], s['end'])}
    if not pred_i and not ref_i:
        return 1.
    return len(ref_i & pred_i) / len(ref_i | pred_i)


def evaluate(refs, preds):
    refs = sorted(refs, key=lambda r: r['id'])
    preds = sorted(preds, key=lambda r: r['id'])
    assert [r['id'] for r in refs] == [p['id'] for p in preds]
    return {
        'Cor':     float(np.mean([score_cor(r, p)     for r, p in zip(refs, preds)])),
        'Cor_lbl': float(np.mean([score_cor_lbl(r, p) for r, p in zip(refs, preds)])),
        'IoU':     float(np.mean([score_iou(r, p)     for r, p in zip(refs, preds)])),
    }

In [6]:
# ── Cell 3: data ─────────────────────────────────────────────────────────────
def load(lang, split_name):
    kind = "labeled" if split_name == "train" else "unlabeled"
    path = f"{DISTRIB}/shroom-vision.{split_name}.{lang}.{kind}.jsonl"
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh]


def split_dev(rows, fraction=0.2, seed=SEED):
    """Stratified split preserving the clean / hallucinated ratio."""
    clean = [r for r in rows if not r["labels"]]
    dirty = [r for r in rows if r["labels"]]
    rng = random.Random(seed)
    rng.shuffle(clean); rng.shuffle(dirty)
    nc, nd = int(len(clean) * fraction), int(len(dirty) * fraction)
    dev = clean[:nc] + dirty[:nd]
    train = clean[nc:] + dirty[nd:]
    rng.shuffle(dev); rng.shuffle(train)
    return train, dev


def char_targets(row):
    n = len(row["response"])
    prob = np.zeros(n, dtype=np.float32)
    cat = np.full(n, -1, dtype=np.int64)
    for span in row.get("labels", []):
        for i in range(span["start"], min(span["end"], n)):
            if span["prob"] >= prob[i]:
                prob[i] = span["prob"]
                cat[i] = CATEGORIES.index(span["label"])
    return prob, cat


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)


def encode(row):
    """Tokenize response (segment 0) with the prompt as context (segment 1)."""
    enc = tokenizer(row["response"], text_pair=row["prompt"],
                    return_offsets_mapping=True, truncation="only_first",
                    max_length=MAX_LEN)
    seq_ids = enc.sequence_ids()
    keep = [i for i, (a, b) in enumerate(enc["offset_mapping"])
            if b > a and seq_ids[i] == 0]
    return enc, keep


class SpanData(Dataset):
    def __init__(self, rows, labeled=True):
        self.rows, self.labeled = rows, labeled
        # Tokenize once up front. Doing it per access pins the CPU at 100% and
        # starves the GPU, since the loader runs in the main process.
        self.cache = [encode(r) for r in rows]

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        row = self.rows[i]
        enc, keep = self.cache[i]
        item = {
            "input_ids": torch.tensor(enc["input_ids"]),
            "attention_mask": torch.tensor(enc["attention_mask"]),
            "keep": torch.tensor(keep, dtype=torch.long),
        }
        if self.labeled:
            prob, cat = char_targets(row)
            offs = [enc["offset_mapping"][i] for i in keep]
            # per token: max character probability, and the category at the peak
            tp = [float(prob[a:b].max()) if b > a else 0.0 for a, b in offs]
            tc = []
            for a, b in offs:
                seg = cat[a:b]
                seg = seg[seg >= 0]
                tc.append(int(np.bincount(seg).argmax()) if len(seg) else -100)
            item["tok_prob"] = torch.tensor(tp, dtype=torch.float)
            item["tok_cat"] = torch.tensor(tc, dtype=torch.long)
        return item


def collate(batch):
    pad = tokenizer.pad_token_id
    maxlen = max(len(b["input_ids"]) for b in batch)
    maxkeep = max(len(b["keep"]) for b in batch)
    out = {
        "input_ids": torch.full((len(batch), maxlen), pad, dtype=torch.long),
        "attention_mask": torch.zeros((len(batch), maxlen), dtype=torch.long),
        "keep": torch.zeros((len(batch), maxkeep), dtype=torch.long),
        "keep_mask": torch.zeros((len(batch), maxkeep), dtype=torch.bool),
    }
    has_labels = "tok_prob" in batch[0]
    if has_labels:
        out["tok_prob"] = torch.zeros((len(batch), maxkeep))
        out["tok_cat"] = torch.full((len(batch), maxkeep), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        L, K = len(b["input_ids"]), len(b["keep"])
        out["input_ids"][i, :L] = b["input_ids"]
        out["attention_mask"][i, :L] = b["attention_mask"]
        out["keep"][i, :K] = b["keep"]
        out["keep_mask"][i, :K] = True
        if has_labels:
            out["tok_prob"][i, :K] = b["tok_prob"]
            out["tok_cat"][i, :K] = b["tok_cat"]
    return out

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [7]:
# ── Cell 4: model ────────────────────────────────────────────────────────────
class SpanTagger(nn.Module):
    """One shared encoder, two token-level heads.

    span head  -> is this token hallucinated        (drives Cor)
    cat  head  -> which of the five categories      (drives Cor_lbl only)
    """

    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_ID)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.span_head = nn.Linear(h, 1)
        self.cat_head = nn.Linear(h, len(CATEGORIES))

    def forward(self, input_ids, attention_mask, keep, keep_mask):
        hidden = self.encoder(input_ids=input_ids,
                              attention_mask=attention_mask).last_hidden_state
        idx = keep.unsqueeze(-1).expand(-1, -1, hidden.size(-1))
        tok = self.dropout(torch.gather(hidden, 1, idx))
        return self.span_head(tok).squeeze(-1), self.cat_head(tok)


def run_epoch(model, loader, optimizer=None, scheduler=None, log_every=50):
    train = optimizer is not None
    model.train() if train else model.eval()
    bce = nn.BCEWithLogitsLoss(reduction="none")
    ce = nn.CrossEntropyLoss(ignore_index=-100)
    total, nb = 0.0, 0
    for step, batch in enumerate(loader, 1):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.set_grad_enabled(train):
            span_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                          batch["keep"], batch["keep_mask"])
            m = batch["keep_mask"]
            # soft targets: BCE against the annotator probability itself
            l_span = (bce(span_logit, batch["tok_prob"]) * m).sum() / m.sum().clamp(min=1)
            l_cat = ce(cat_logit.reshape(-1, len(CATEGORIES)), batch["tok_cat"].reshape(-1))
            loss = l_span + 0.5 * l_cat
        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total += loss.item(); nb += 1
        if train and step % log_every == 0:
            print(f"  step {step}/{len(loader)}  loss {total / nb:.4f}", flush=True)
    return total / max(nb, 1)

In [8]:
# ── Cell 5: inference ────────────────────────────────────────────────────────
@torch.no_grad()
def predict_char_probs(model, rows):
    """Return per-character probability and category arrays for each row."""
    model.eval()
    dataset = SpanData(rows, labeled=False)
    loader = DataLoader(dataset, batch_size=BATCH, shuffle=False, collate_fn=collate)
    results, cursor = [], 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        span_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                      batch["keep"], batch["keep_mask"])
        probs = torch.sigmoid(span_logit).cpu().numpy()
        cats = cat_logit.argmax(-1).cpu().numpy()
        for i in range(len(probs)):
            row = rows[cursor]
            enc, keep = dataset.cache[cursor]
            cursor += 1
            offs = [enc["offset_mapping"][k] for k in keep]
            n = len(row["response"])
            cp = np.zeros(n, dtype=np.float32)
            cc = np.zeros(n, dtype=np.int64)
            for j, (a, b) in enumerate(offs):
                cp[a:min(b, n)] = probs[i][j]
                cc[a:min(b, n)] = cats[i][j]
            results.append((cp, cc))
    return results


def to_spans(char_prob, char_cat, threshold):
    """Contiguous above-threshold runs, split where the category changes."""
    spans, n, i = [], len(char_prob), 0
    while i < n:
        if char_prob[i] < threshold:
            i += 1; continue
        j, c = i, char_cat[i]
        while j < n and char_prob[j] >= threshold and char_cat[j] == c:
            j += 1
        spans.append({"start": int(i), "end": int(j),
                      "prob": float(round(float(np.mean(char_prob[i:j])), 6)),
                      "label": CATEGORIES[int(c)]})
        i = j
    return spans


def build_preds(rows, char_preds, threshold):
    return [{"id": r["id"], "labels": to_spans(cp, cc, threshold)}
            for r, (cp, cc) in zip(rows, char_preds)]


def tune_threshold(dev_rows, char_preds):
    """Pick the threshold maximizing Cor on dev. Cor is the reported metric."""
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in dev_rows]
    best = (None, -1, None)
    for t in np.arange(0.10, 0.91, 0.05):
        s = evaluate(refs, build_preds(dev_rows, char_preds, float(t)))
        if s["Cor"] > best[1]:
            best = (float(t), s["Cor"], s)
    return best

In [9]:
# ── Cell 6: train ────────────────────────────────────────────────────────────
train_rows, dev_rows = [], []
for lang in LANGS:
    tr, dv = split_dev(load(lang, "train"))
    train_rows += tr; dev_rows += dv
print(f"train={len(train_rows)}  dev={len(dev_rows)}")

model = SpanTagger().to(DEVICE)
train_loader = DataLoader(SpanData(train_rows), batch_size=BATCH, shuffle=True,
                          collate_fn=collate)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer, int(0.1 * len(train_loader) * EPOCHS), len(train_loader) * EPOCHS)

for epoch in range(EPOCHS):
    loss = run_epoch(model, train_loader, optimizer, scheduler)
    print(f"epoch {epoch + 1}: loss {loss:.4f}", flush=True)
    # checkpoint every epoch so a disconnect costs one epoch, not the whole run
    torch.save(model.state_dict(), f"{OUT_DIR}/span_tagger.pt")

train=12085  dev=3017


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-large
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  step 50/1511  loss 1.7501
  step 100/1511  loss 1.5787
  step 150/1511  loss 1.3607
  step 200/1511  loss 1.2311
  step 250/1511  loss 1.1343
  step 300/1511  loss 1.0808
  step 350/1511  loss 1.0275
  step 400/1511  loss 0.9984
  step 450/1511  loss 0.9711
  step 500/1511  loss 0.9485
  step 550/1511  loss 0.9250
  step 600/1511  loss 0.9080
  step 650/1511  loss 0.8906
  step 700/1511  loss 0.8772
  step 750/1511  loss 0.8652
  step 800/1511  loss 0.8532
  step 850/1511  loss 0.8461
  step 900/1511  loss 0.8354
  step 950/1511  loss 0.8256
  step 1000/1511  loss 0.8204
  step 1050/1511  loss 0.8153
  step 1100/1511  loss 0.8075
  step 1150/1511  loss 0.8000
  step 1200/1511  loss 0.7954
  step 1250/1511  loss 0.7899
  step 1300/1511  loss 0.7833
  step 1350/1511  loss 0.7796
  step 1400/1511  loss 0.7746
  step 1450/1511  loss 0.7696
  step 1500/1511  loss 0.7659
epoch 1: loss 0.7660
  step 50/1511  loss 0.5995
  step 100/1511  loss 0.6048
  step 150/1511  loss 0.6128
  step 200/15

In [10]:
# ── Cell 7: evaluate per language ────────────────────────────────────────────
print(f"\n{'lang':<6}{'thr':>6}{'Cor':>9}{'Cor_lbl':>9}{'IoU':>9}   (mark_none Cor)")
thresholds = {}
for lang in LANGS:
    rows = [r for r in dev_rows if r["language"] == lang]
    cps = predict_char_probs(model, rows)
    thr, cor, scores = tune_threshold(rows, cps)
    thresholds[lang] = thr
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in rows]
    floor = evaluate(refs, [{"id": r["id"], "labels": []} for r in rows])["Cor"]
    print(f"{lang:<6}{thr:>6.2f}{scores['Cor']:>9.3f}{scores['Cor_lbl']:>9.3f}"
          f"{scores['IoU']:>9.3f}   {floor:.3f}")



lang     thr      Cor  Cor_lbl      IoU   (mark_none Cor)
en      0.30    0.364    0.318    0.323   0.253
fr      0.25    0.382    0.326    0.339   0.255
it      0.20    0.416    0.352    0.375   0.263
zh      0.30    0.434    0.399    0.409   0.365


In [11]:
# ── Cell 8: predict test and write submission ────────────────────────────────
for lang in LANGS:
    rows = load(lang, "test")
    cps = predict_char_probs(model, rows)
    preds = build_preds(rows, cps, thresholds[lang])
    path = f"{OUT_DIR}/predictions_{lang}.jsonl"
    with open(path, "w", encoding="utf-8") as fh:
        for p in preds:
            fh.write(json.dumps(p, ensure_ascii=False) + "\n")
    n_spans = sum(len(p["labels"]) for p in preds)
    n_empty = sum(1 for p in preds if not p["labels"])
    print(f"{lang}: {len(preds)} rows, {n_spans} spans, {n_empty} empty -> {path}")


en: 1201 rows, 2557 spans, 555 empty -> /kaggle/working/predictions_en.jsonl
fr: 1233 rows, 4180 spans, 453 empty -> /kaggle/working/predictions_fr.jsonl
it: 1254 rows, 3088 spans, 394 empty -> /kaggle/working/predictions_it.jsonl
zh: 1210 rows, 944 spans, 715 empty -> /kaggle/working/predictions_zh.jsonl


In [12]:
!wget -q https://a3s.fi/mickusti-2007780-pub/participant_kit.shroom_visions.zip -O /tmp/kit.zip
!unzip -oq /tmp/kit.zip -d /tmp/kit
!cd /kaggle/working && python /tmp/kit/participant_kit/format_checker.py predictions_en.jsonl predictions_fr.jsonl predictions_it.jsonl predictions_zh.jsonl --reference-dir /kaggle/working/distrib

Checked 4 file(s), 4898 row(s), 10769 span(s).
Languages: en, fr, it, zh
OK: submission format looks valid.


In [13]:
from IPython.display import FileLink
!cd /kaggle/working && zip -q predictions.zip predictions_en.jsonl predictions_fr.jsonl predictions_it.jsonl predictions_zh.jsonl
FileLink('predictions.zip')

/kaggle/working/predictions.zip

In [14]:
# ── Cell 9: category assignment strategies (no retraining needed) ────────────
#
# The span head stays exactly as trained. Only the way we turn per-token
# category scores into span labels changes. Compares three strategies on dev.

import numpy as np
import torch
from torch.utils.data import DataLoader


@torch.no_grad()
def predict_char_full(model, rows):
    """Per-character hallucination probability plus the full category distribution.

    Returns (char_prob, char_cat_probs) per row, where char_cat_probs has shape
    (n_chars, len(CATEGORIES)). Keeping the distribution instead of an argmax is
    what lets us pool categories over a span or a whole response.
    """
    model.eval()
    dataset = SpanData(rows, labeled=False)
    loader = DataLoader(dataset, batch_size=BATCH, shuffle=False, collate_fn=collate)
    results, cursor = [], 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        span_logit, cat_logit = model(batch["input_ids"], batch["attention_mask"],
                                      batch["keep"], batch["keep_mask"])
        probs = torch.sigmoid(span_logit).cpu().numpy()
        cat_p = torch.softmax(cat_logit, dim=-1).cpu().numpy()
        for i in range(len(probs)):
            row = rows[cursor]
            enc, keep = dataset.cache[cursor]
            cursor += 1
            offs = [enc["offset_mapping"][k] for k in keep]
            n = len(row["response"])
            cp = np.zeros(n, dtype=np.float32)
            cd = np.zeros((n, len(CATEGORIES)), dtype=np.float32)
            for j, (a, b) in enumerate(offs):
                e = min(b, n)
                cp[a:e] = probs[i][j]
                cd[a:e] = cat_p[i][j]
            results.append((cp, cd))
    return results


def spans_with_strategy(char_prob, char_dist, threshold, strategy):
    """Build spans, assigning categories by one of three strategies.

    per_token : argmax at each character; a span breaks where the label changes
    per_span  : one label per contiguous run, pooled over its characters
    per_resp  : one label for the whole response, pooled over flagged characters
    """
    n = len(char_prob)
    above = char_prob >= threshold
    if not above.any():
        return []

    resp_label = int(char_dist[above].sum(axis=0).argmax())

    # contiguous runs of flagged characters
    runs, i = [], 0
    while i < n:
        if not above[i]:
            i += 1
            continue
        j = i
        while j < n and above[j]:
            j += 1
        runs.append((i, j))
        i = j

    spans = []
    for a, b in runs:
        if strategy == "per_resp":
            pieces = [(a, b, resp_label)]
        elif strategy == "per_span":
            pieces = [(a, b, int(char_dist[a:b].sum(axis=0).argmax()))]
        else:  # per_token — split the run wherever the argmax label changes
            labels = char_dist[a:b].argmax(axis=1)
            pieces, s = [], 0
            for k in range(1, len(labels) + 1):
                if k == len(labels) or labels[k] != labels[s]:
                    pieces.append((a + s, a + k, int(labels[s])))
                    s = k
        for s, e, lab in pieces:
            spans.append({"start": int(s), "end": int(e),
                          "prob": float(round(float(char_prob[s:e].mean()), 6)),
                          "label": CATEGORIES[lab]})
    return spans


def eval_strategy(rows, char_preds, threshold, strategy):
    refs = [{"id": r["id"], "labels": r["labels"], "text_len": len(r["response"])}
            for r in rows]
    preds = [{"id": r["id"],
              "labels": spans_with_strategy(cp, cd, threshold, strategy)}
             for r, (cp, cd) in zip(rows, char_preds)]
    scores = evaluate(refs, preds)
    counts = [len({s["label"] for s in p["labels"]}) for p in preds if p["labels"]]
    scores["cats_per_resp"] = float(np.mean(counts)) if counts else 0.0
    scores["n_flagged"] = len(counts)
    return scores


# ── run the comparison ───────────────────────────────────────────────────────
STRATEGIES = ["per_token", "per_span", "per_resp"]
best_strategy, best_thr = {}, {}

print(f"{'lang':<5}{'strategy':<12}{'thr':>6}{'Cor':>9}{'Cor_lbl':>10}{'cats/resp':>11}")
print("-" * 53)
for lang in LANGS:
    rows = [r for r in dev_rows if r["language"] == lang]
    preds_full = predict_char_full(model, rows)
    best = (None, None, -1)
    for strategy in STRATEGIES:
        top = (None, -1, None)
        for t in np.arange(0.10, 0.91, 0.05):
            s = eval_strategy(rows, preds_full, float(t), strategy)
            # Cor+Lbl is listed first on the leaderboard, so tune for it
            if s["Cor_lbl"] > top[1]:
                top = (float(t), s["Cor_lbl"], s)
        thr, cl, s = top
        print(f"{lang:<5}{strategy:<12}{thr:>6.2f}{s['Cor']:>9.3f}"
              f"{s['Cor_lbl']:>10.3f}{s['cats_per_resp']:>11.2f}")
        if cl > best[2]:
            best = (strategy, thr, cl)
    best_strategy[lang], best_thr[lang] = best[0], best[1]
    print(f"{'':5}-> best: {best[0]} @ {best[1]:.2f}  (Cor_lbl {best[2]:.3f})")
    print()

print("chosen:", {l: (best_strategy[l], round(best_thr[l], 2)) for l in LANGS})

lang strategy       thr      Cor   Cor_lbl  cats/resp
-----------------------------------------------------
en   per_token     0.45    0.353     0.324       1.03
en   per_span      0.45    0.353     0.324       1.02
en   per_resp      0.45    0.353     0.324       1.00
     -> best: per_resp @ 0.45  (Cor_lbl 0.324)

fr   per_token     0.25    0.382     0.326       1.22
fr   per_span      0.40    0.361     0.326       1.09
fr   per_resp      0.40    0.361     0.324       1.00
     -> best: per_span @ 0.40  (Cor_lbl 0.326)

it   per_token     0.25    0.413     0.355       1.20
it   per_span      0.25    0.413     0.355       1.18
it   per_resp      0.30    0.405     0.353       1.00
     -> best: per_span @ 0.25  (Cor_lbl 0.355)

zh   per_token     0.40    0.432     0.409       1.05
zh   per_span      0.40    0.432     0.409       1.04
zh   per_resp      0.40    0.432     0.409       1.00
     -> best: per_span @ 0.40  (Cor_lbl 0.409)

chosen: {'en': ('per_resp', 0.45), 'fr': ('per_span'